## Required libraries installation

In [1]:
%pip install accelerate scipy librosa -q
%pip install transformers torchaudio pandas datasets -q
%pip install soundfile -q
%pip install transformers==4.44.2 -q
%pip install tqdm -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## All imports and custom built modules and functions

In [2]:
# Normal imports
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Torch imports
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor


/home/warden/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device agnostic code

In [3]:
! nvidia-smi

Wed Oct 14 21:30:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.04              KMD Version: 610.57.04     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              2W /  115W |       5MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
device = "cuda" if torch.cuda.is_available else "cpu"
device
# if it shows cpu download cuda torch audio

'cuda'

### Inspecting the data on some nonchalant shii

In [5]:
# Define paths matching your project layout
DEV_CSV = "dataset/FSD50K.ground_truth/dev.csv"
TRAIN_DIR = "split_dataset/train"
TEST_DIR = "split_dataset/test"

# 1. Inspect CSV Headers
df_dev = pd.read_csv(DEV_CSV)
print("=== CSV Column Headers ===")
print(df_dev.columns.tolist())
print("\n=== First 3 Rows ===")
print(df_dev.head(3))

# 2. Verify Audio File Counts on Disk
train_wavs = glob.glob(os.path.join(TRAIN_DIR, "*.wav"))
test_wavs = glob.glob(os.path.join(TEST_DIR, "*.wav"))

print(f"\nFound {len(train_wavs)} .wav files in '{TRAIN_DIR}'")
print(f"Found {len(test_wavs)} .wav files in '{TEST_DIR}'")

=== CSV Column Headers ===
['fname', 'labels', 'mids', 'split']

=== First 3 Rows ===
   fname                                             labels  \
0  64760  Electric_guitar,Guitar,Plucked_string_instrume...   
1  16399  Electric_guitar,Guitar,Plucked_string_instrume...   
2  16401  Electric_guitar,Guitar,Plucked_string_instrume...   

                                            mids  split  
0  /m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf  train  
1  /m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf  train  
2  /m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf  train  

Found 1580 .wav files in 'split_dataset/train'
Found 395 .wav files in 'split_dataset/test'


In [6]:
class FSD50KThreatDataset(Dataset):
    def __init__(
        self,
        audio_dir,
        csv_path,
        target_classes=["chainsaw", "gunshot", "engine", "background"],
        target_duration=10,
        sample_rate=16000,
    ):
        self.audio_dir = audio_dir
        self.target_classes = target_classes
        self.sample_rate = sample_rate
        self.target_length = target_duration * sample_rate  # 160,000 samples

        # 1. Filter CSV to ONLY keep files that actually exist in your audio_dir on disk
        raw_df = pd.read_csv(csv_path)
        existing_rows = []

        print("Filtering CSV against existing disk files...")
        for _, row in raw_df.iterrows():
            fname = str(row["fname"])
            if not fname.endswith(".wav"):
                fname += ".wav"
            if os.path.exists(os.path.join(audio_dir, fname)):
                existing_rows.append(row)

        self.df = pd.DataFrame(existing_rows).reset_index(drop=True)
        print(
            f"Dataset Ready: Filtered from {len(raw_df)} metadata rows down to {len(self.df)} existing local files."
        )

        # Load AST feature extractor once
        self.feature_extractor = ASTFeatureExtractor.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )

    def _load_and_standardize(self, file_path: str) -> torch.Tensor:
        waveform, sr = torchaudio.load(file_path)

        # Downmix Stereo -> Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample -> 16 kHz
        if sr != self.sample_rate:
            resampler = T.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        # Pad / Truncate -> 10 seconds (160,000 samples)
        if waveform.shape[-1] < self.target_length:
            padding = self.target_length - waveform.shape[-1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, : self.target_length]

        return waveform

    def _get_multi_hot_label(self, raw_label: str) -> torch.Tensor:
        multi_hot = torch.zeros(len(self.target_classes), dtype=torch.float32)
        raw_label_str = str(raw_label).lower()

        matched = False
        for idx, class_name in enumerate(self.target_classes):
            if class_name.lower() in raw_label_str:
                multi_hot[idx] = 1.0
                matched = True

        if not matched and "background" in self.target_classes:
            bg_idx = self.target_classes.index("background")
            multi_hot[bg_idx] = 1.0

        return multi_hot

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        fname = str(row["fname"])
        if not fname.endswith(".wav"):
            fname += ".wav"

        file_path = os.path.join(self.audio_dir, fname)
        waveform = self._load_and_standardize(file_path)

        inputs = self.feature_extractor(
            waveform.squeeze(0).numpy(),
            sampling_rate=self.sample_rate,
            return_tensors="pt",
        )

        label_vec = self._get_multi_hot_label(row["labels"])

        return {"input_values": inputs.input_values.squeeze(0), "labels": label_vec}


# --- CRITICAL FIX: Set num_workers=0 inside Jupyter Notebooks on Windows ---
TARGET_CLASSES = ["chainsaw", "gunshot", "engine", "background"]

train_dataset = FSD50KThreatDataset(
    audio_dir="split_dataset/train",
    csv_path="dataset/FSD50K.ground_truth/dev.csv",
    target_classes=TARGET_CLASSES,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,  # Set to 0 to prevent Windows notebook multiprocessing deadlocks
    pin_memory=False,
)

# Test single batch extraction
print("\nFetching first batch...")
batch = next(iter(train_loader))
print("Input Spectrogram Batch Shape:", batch["input_values"].shape)  # [8, 1024, 128]
print("Multi-Hot Label Batch Shape  :", batch["labels"].shape)  # [8, 4]

Filtering CSV against existing disk files...
Dataset Ready: Filtered from 40966 metadata rows down to 1580 existing local files.

Fetching first batch...
Input Spectrogram Batch Shape: torch.Size([2, 1024, 128])
Multi-Hot Label Batch Shape  : torch.Size([2, 4])


In [7]:
train_loader

In [8]:
import torch
from training import train_ast_model, ForestThreatAST

TARGET_CLASSES = ["chainsaw", "gunshot", "engine", "background"]

model = ForestThreatAST(num_classes=len(TARGET_CLASSES)).to(device)

loading configuration file config.json from cache at /home/warden/.cache/huggingface/hub/models--MIT--ast-finetuned-audioset-10-10-0.4593/snapshots/f826b80d28226b62986cc218e5cec390b1096902/config.json
Model config ASTConfig {
  "architectures": [
    "ASTForAudioClassification"
  ],
  "attention_probs_dropout_prob": 0.0,
  "frequency_stride": 10,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "id2label": {
    "0": "Speech",
    "1": "Male speech, man speaking",
    "2": "Female speech, woman speaking",
    "3": "Child speech, kid speaking",
    "4": "Conversation",
    "5": "Narration, monologue",
    "6": "Babbling",
    "7": "Speech synthesizer",
    "8": "Shout",
    "9": "Bellow",
    "10": "Whoop",
    "11": "Yell",
    "12": "Battle cry",
    "13": "Children shouting",
    "14": "Screaming",
    "15": "Whispering",
    "16": "Laughter",
    "17": "Baby laughter",
    "18": "Giggle",
    "19": "Snicker",
    "20": "Belly laugh",
    "21": "Chuckle, 

In [ ]:
EPOCHS = 2
LEARNING_RATE = 1e-5
SAVE_DIR = "trained_models"

trained_model = train_ast_model(
    model=model,
    train_loader=train_loader,
    device=device,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    save_path=f"{SAVE_DIR}/ast_best_model.pth",
)

Executing AST Fine-Tuning on device: CUDA


Epoch [2/2]: 100%|██████████| 790/790 [01:50<00:00,  7.15it/s, avg_loss=0.0162]



Training Complete! Checkpoint saved as 'trained_models/ast_best_model.pth'.
